In [1]:
# ================================
# exp28_model_ensemble
# ElasticNet + XGBoost ensemble
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor


def metric_diagnostics(y_true,y_pred):

    rmse=np.sqrt(mean_squared_error(y_true,y_pred))
    mae=mean_absolute_error(y_true,y_pred)

    y_true_log=np.log1p(y_true)
    y_pred_log=np.log1p(np.maximum(y_pred,0))
    rmsle=np.sqrt(mean_squared_error(y_true_log,y_pred_log))

    nrmse_mean=rmse/np.mean(y_true)
    nrmse_range=rmse/(np.max(y_true)-np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:",rmse)
    print("MAE:",mae)
    print("RMSLE:",rmsle)
    print("NRMSE (mean):",nrmse_mean)
    print("NRMSE (range):",nrmse_range)


train=pd.read_csv("../data/train.csv",encoding="cp932")
test=pd.read_csv("../data/test.csv",encoding="cp932")

target="含水率"
id_col="sample number"

spectral_cols=[c for c in train.columns if c not in ["sample number","species number","樹種","含水率"]]

X=train[spectral_cols]
y=train[target]
X_test=test[spectral_cols]


kf=KFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
test_pred=np.zeros(len(X_test))


elastic=Pipeline([
("scaler",StandardScaler()),
("model",ElasticNet(alpha=0.001,l1_ratio=0.9,max_iter=50000))
])

xgb=XGBRegressor(
n_estimators=800,
max_depth=4,
learning_rate=0.04,
subsample=0.8,
colsample_bytree=0.8,
objective="reg:squarederror",
random_state=42,
n_jobs=-1
)


for fold,(train_idx,val_idx) in enumerate(kf.split(X)):

    X_train,X_val=X.iloc[train_idx],X.iloc[val_idx]
    y_train,y_val=y.iloc[train_idx],y.iloc[val_idx]

    elastic.fit(X_train,y_train)
    xgb.fit(X_train,y_train)

    pred_elastic=elastic.predict(X_val)
    pred_xgb=xgb.predict(X_val)

    pred=(pred_elastic+pred_xgb)/2

    oof[val_idx]=pred

    test_pred+=(elastic.predict(X_test)+xgb.predict(X_test))/2/kf.n_splits


print("\nFinal Ensemble Performance")
metric_diagnostics(y,oof)


os.makedirs("../submissions",exist_ok=True)

submission=pd.DataFrame({
    id_col:test[id_col],
    target:test_pred
})

output_path="../submissions/exp28_model_ensemble.csv"

submission.to_csv(output_path,index=False,header=False)

print("\nSubmission saved:",output_path)

/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.304e+05, tolerance: 2.646e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.251e+05, tolerance: 2.477e+02
  model = cd_fast.enet_coordinate_descent(
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check t


Final Ensemble Performance

Metric diagnostics
------------------
RMSE: 12.720615292636973
MAE: 6.944805203995472
RMSLE: 0.2762075281772401
NRMSE (mean): 0.2547298417070052
NRMSE (range): 0.042723728728759555

Submission saved: ../submissions/exp28_model_ensemble.csv
